# Asymptotic variances in the Ornstein-Uhlenbeck setting

This notebook computes the table of standard deviations that accompanies the central limit theorem for the three coefficient regimes
(population coefficients P, coefficients from an independent sample I, coefficients from the transported sample S).

**Setting.** Target $N(0,1)$, source $N(1.5,0.5^2)$, analytical Hermite eigenpairs $\lambda_k=k$, $\varphi_k=He_k/\sqrt{k!}$,
$M=M'=2000$ particles, $T=6$, fourth-order Runge-Kutta steps $h=0.025$, threshold $\varepsilon_0=10^{-3}$, velocity bound $v_{\max}=20$,
ranks $r=20$ and $40$, $1000$ realizations of the source, test functions $\sin x$, $\tanh x$ and $e^{-x^2}$.

**Observed values.** Standard deviation of the particle mean over the realizations.

**Asymptotic values.** $\sigma/\sqrt M$ with the variances of the theorem, computed by quadrature over the source on a uniform grid of
$800001$ points. The gradient of $m$ at the population coefficients is the integral over the source of the derivative of
$f(X_T)$ with respect to the coefficients, and this derivative is obtained by differentiating the Runge-Kutta scheme.

The full computation takes about 40 minutes on two cores. With `RUN = False` the tables are rebuilt from the results stored in the notebook,
which were produced by the code below.

In [1]:
RUN = False   # True recomputes everything (about 40 minutes on two cores)
"""Asymptotic variances of Theorem 4.7 in the Ornstein-Uhlenbeck setting (Table 2 of the paper).

Target N(0,1), source N(1.5, 0.5^2), analytical Hermite eigenpairs, M = M' = 2000 particles, T = 6, RK4 step 0.025,
threshold 1e-3, velocity bound 20, ranks 20 and 40, test functions sin x, tanh x, exp(-x^2),
coefficient regimes (P) population, (I) independent sample, (S) same sample.

Observed values: standard deviation of the particle mean over NREP realizations of the source.
Asymptotic values: sigma / sqrt(M) with the variances of Theorem 4.7, computed by quadrature over the source on a uniform grid.
The gradient of m at the population coefficients is the integral over the source of the derivative of f(X_T) with respect to
the coefficients, and this derivative is obtained by differentiating the RK4 scheme (linearized flow)."""
import json, math, sys, time
import numpy as np
from numba import njit, prange

M0, S0, T, H, EPS0, VMAX, M = 1.5, 0.5, 6.0, 0.025, 1e-3, 20.0, 2000
NREP, NGRID, RANKS = 1000, 800_001, (20, 40)


@njit
def velocity(x, w, r):
    """Safeguarded velocity -rho'/max(rho, eps0), clipped at vmax, for rho = sum_k w_k h_k."""
    hp, h = 1.0, x                       # h_0, h_1 (normalized Hermite polynomials He_k / sqrt(k!))
    rho = w[0] + w[1] * x
    drho = w[1]
    for k in range(1, r):
        hn = (x * h - math.sqrt(k) * hp) / math.sqrt(k + 1.0)
        drho += w[k + 1] * math.sqrt(k + 1.0) * h
        rho += w[k + 1] * hn
        hp, h = h, hn
    v = -drho / max(rho, EPS0)
    return min(max(v, -VMAX), VMAX)


@njit(parallel=True)
def transport(x0, c, r, nsteps, h):
    """RK4 for dx/dt = v_t(x) with coefficients c_k exp(-k t); the particles are not confined."""
    out = np.empty_like(x0)
    lam = np.arange(r + 1).astype(np.float64)
    for j in prange(x0.shape[0]):
        x = x0[j]
        for i in range(nsteps):
            t = i * h
            w0 = c * np.exp(-lam * t); w1 = c * np.exp(-lam * (t + 0.5 * h)); w2 = c * np.exp(-lam * (t + h))
            k1 = velocity(x, w0, r)
            k2 = velocity(x + 0.5 * h * k1, w1, r)
            k3 = velocity(x + 0.5 * h * k2, w1, r)
            k4 = velocity(x + h * k3, w2, r)
            x += h / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)
        out[j] = x
    return out


@njit
def velocity_lin(x, c, e, r, hb, vc):
    """Velocity, its derivative in x and (in vc) its derivatives in c_1..c_r, with e_k = exp(-k t). hb is a work array."""
    hb[0] = 1.0; hb[1] = x
    for k in range(1, r):
        hb[k + 1] = (x * hb[k] - math.sqrt(k) * hb[k - 1]) / math.sqrt(k + 1.0)
    rho = 0.0; d1 = 0.0; d2 = 0.0
    for k in range(r + 1):
        w = c[k] * e[k]
        rho += w * hb[k]
        if k >= 1: d1 += w * math.sqrt(k) * hb[k - 1]
        if k >= 2: d2 += w * math.sqrt(k * (k - 1.0)) * hb[k - 2]
    if rho >= EPS0:
        raw = -d1 / rho; vx = -d2 / rho + raw * raw
        for k in range(1, r + 1):
            vc[k - 1] = e[k] * (-math.sqrt(k) * hb[k - 1] - raw * hb[k]) / rho
    else:
        raw = -d1 / EPS0; vx = -d2 / EPS0
        for k in range(1, r + 1):
            vc[k - 1] = -e[k] * math.sqrt(k) * hb[k - 1] / EPS0
    if raw > VMAX or raw < -VMAX:
        raw = VMAX if raw > 0 else -VMAX; vx = 0.0
        for k in range(r): vc[k] = 0.0
    return raw, vx


@njit(parallel=True)
def transport_lin(x0, c, r, nsteps, h):
    """RK4 flow and its derivative with respect to the coefficients c_1..c_r (derivative of the RK4 scheme)."""
    n = x0.shape[0]
    xT = np.empty(n); sT = np.empty((n, r))
    lam = np.arange(r + 1).astype(np.float64)
    for j in prange(n):
        x = x0[j]; s = np.zeros(r); hb = np.empty(r + 1)
        b1 = np.empty(r); b2 = np.empty(r); b3 = np.empty(r); b4 = np.empty(r)
        for i in range(nsteps):
            t = i * h
            e0 = np.exp(-lam * t); e1 = np.exp(-lam * (t + 0.5 * h)); e2 = np.exp(-lam * (t + h))
            k1, a1 = velocity_lin(x, c, e0, r, hb, b1)
            d1 = a1 * s + b1
            k2, a2 = velocity_lin(x + 0.5 * h * k1, c, e1, r, hb, b2)
            d2 = a2 * (s + 0.5 * h * d1) + b2
            k3, a3 = velocity_lin(x + 0.5 * h * k2, c, e1, r, hb, b3)
            d3 = a3 * (s + 0.5 * h * d2) + b3
            k4, a4 = velocity_lin(x + h * k3, c, e2, r, hb, b4)
            d4 = a4 * (s + h * d3) + b4
            x += h / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)
            s += h / 6.0 * (d1 + 2 * d2 + 2 * d3 + d4)
        xT[j] = x; sT[j] = s
    return xT, sT


def hermite(x, r):
    """Rows h_0..h_r of the normalized Hermite polynomials He_k / sqrt(k!) at x."""
    P = np.empty((r + 1, x.size)); P[0] = 1.0
    if r >= 1: P[1] = x
    for k in range(1, r):
        P[k + 1] = (x * P[k] - math.sqrt(k) * P[k - 1]) / math.sqrt(k + 1.0)
    return P


def population_coefficients(r):
    """Averages of h_0..h_r over the source N(m0, s0^2), in closed form."""
    q = 1.0 - S0 ** 2
    return hermite(np.array([M0 / math.sqrt(q)]), r)[:, 0] * q ** (0.5 * np.arange(r + 1))


FUN = {'sin x': (np.sin, np.cos), 'tanh x': (np.tanh, lambda x: 1.0 - np.tanh(x) ** 2),
       'exp(-x^2)': (lambda x: np.exp(-x * x), lambda x: -2.0 * x * np.exp(-x * x))}
SD_PI = {'sin x': math.sqrt((1 - math.exp(-2)) / 2), 'exp(-x^2)': math.sqrt(1 / math.sqrt(5) - 1 / 3)}
_xq, _wq = np.polynomial.hermite_e.hermegauss(200)
SD_PI['tanh x'] = float(np.sqrt(np.sum(_wq / _wq.sum() * np.tanh(_xq) ** 2)))   # standard deviations of f under the target


def observed(r, nrep, nparticles=M, regimes='PIS'):
    """Standard deviation over nrep realizations of the particle mean of each test function, in the coefficient regimes."""
    nsteps = int(round(T / H)); cbar = population_coefficients(r)
    means = {reg: {f: np.empty(nrep) for f in FUN} for reg in regimes}
    for rep in range(nrep):
        ss = np.random.SeedSequence([2026, r, rep]).spawn(2)
        x0 = M0 + S0 * np.random.default_rng(ss[0]).standard_normal(nparticles)      # transported sample
        y0 = M0 + S0 * np.random.default_rng(ss[1]).standard_normal(nparticles)      # independent sample of regime (I)
        coeff = {'P': cbar, 'I': hermite(y0, r).mean(1), 'S': hermite(x0, r).mean(1)}
        for reg in regimes:
            c = coeff[reg].copy(); c[0] = 1.0
            xT = transport(x0, c, r, nsteps, H)
            for f, (fn, _) in FUN.items(): means[reg][f][rep] = fn(xT).mean()
    return {reg: {f: float(np.std(v, ddof=1)) for f, v in d.items()} for reg, d in means.items()}


def asymptotic(r, ngrid=NGRID, width=9.0):
    """sigma_P, sigma_I, sigma_S of Theorem 4.7 and |beta|, by quadrature over the source."""
    nsteps = int(round(T / H)); cbar = population_coefficients(r)
    x = np.linspace(M0 - width * S0, M0 + width * S0, ngrid)
    w = np.exp(-0.5 * ((x - M0) / S0) ** 2); w /= w.sum()                           # quadrature weights of the source
    xT, sT = transport_lin(x, cbar, r, nsteps, H)                                   # population flow and its derivative in c
    P = hermite(x, r)[1:]                                                           # eigenfunctions 1..r at the source points
    xg = np.linspace(-8.0, 8.0, 8001); wg = np.exp(-0.5 * xg ** 2); wg /= wg.sum()  # quadrature weights of the target
    xgT = transport(xg, cbar, r, nsteps, H); Pg = hermite(xg, r)[1:]
    var = lambda u: float(w @ (u - w @ u) ** 2)
    out = {}
    for f, (fn, dfn) in FUN.items():
        g = fn(xT)
        grad = (w * dfn(xT)) @ sT                                                   # gradient of m at the population coefficients
        lin = grad @ P
        beta = grad + Pg @ (fn(xgT) * wg)                                           # beta = grad m + (<g, phi_k>_pi)_k
        out[f] = dict(P=math.sqrt(var(g)), I=math.sqrt(var(g) + var(lin)), S=math.sqrt(var(g + lin)), beta=float(np.linalg.norm(beta)))
    return out


def table(nrep=NREP):
    """Rows of Table 2, standard deviations in units of 1e-3."""
    rows = []
    for r in RANKS:
        t0 = time.time(); thm = asymptotic(r); t1 = time.time(); obs = observed(r, nrep)
        print(f'r={r}: quadrature {t1 - t0:.0f} s, {nrep} realizations {time.time() - t1:.0f} s', flush=True)
        for f in FUN:
            row = dict(f=f, r=r, independent=1e3 * SD_PI[f] / math.sqrt(M), beta_norm=thm[f]['beta'])
            for reg in 'PIS':
                row[reg + '_obs'] = 1e3 * obs[reg][f]; row[reg + '_thm'] = 1e3 * thm[f][reg] / math.sqrt(M)
            rows.append(row)
    return rows

In [2]:
STORED = {
 "config": {
  "m0": 1.5,
  "s0": 0.5,
  "T": 6.0,
  "h": 0.025,
  "eps0": 0.001,
  "vmax": 20.0,
  "M": 2000,
  "nrep": 1000,
  "ngrid": 800001
 },
 "rows": [
  {
   "f": "sin x",
   "r": 20,
   "independent": 14.702590900614993,
   "beta_norm": 0.2885394143619625,
   "P_obs": 15.13804009158713,
   "P_thm": 14.716270777519007,
   "I_obs": 20.803032391670424,
   "I_thm": 20.893160316027686,
   "S_obs": 0.7497410121690476,
   "S_thm": 0.6779165746703609
  },
  {
   "f": "tanh x",
   "r": 20,
   "independent": 14.040913260857373,
   "beta_norm": 0.003489276137191018,
   "P_obs": 14.432568715990323,
   "P_thm": 14.040270201357501,
   "I_obs": 19.73948187218162,
   "I_thm": 19.8063388023242,
   "S_obs": 0.7409795963202566,
   "S_thm": 0.7653914194008625
  },
  {
   "f": "exp(-x^2)",
   "r": 20,
   "independent": 7.54586847773749,
   "beta_norm": 0.004546429254388661,
   "P_obs": 7.637438545932791,
   "P_thm": 7.571222567865801,
   "I_obs": 9.97904880273483,
   "I_thm": 10.258137116633094,
   "S_obs": 1.9353285481897509,
   "S_thm": 1.9513394921125324
  },
  {
   "f": "sin x",
   "r": 40,
   "independent": 14.702590900614993,
   "beta_norm": 0.0841133326747265,
   "P_obs": 14.594781727072156,
   "P_thm": 14.702234267092477,
   "I_obs": 20.651000110928997,
   "I_thm": 20.784801375894403,
   "S_obs": 0.34153343146101156,
   "S_thm": 0.1598031298330194
  },
  {
   "f": "tanh x",
   "r": 40,
   "independent": 14.040913260857373,
   "beta_norm": 0.001573065411475669,
   "P_obs": 14.171378907745726,
   "P_thm": 14.04113058057842,
   "I_obs": 19.8201715502809,
   "I_thm": 19.84333986401739,
   "S_obs": 0.17651390998656705,
   "S_thm": 0.17024629874516692
  },
  {
   "f": "exp(-x^2)",
   "r": 40,
   "independent": 7.54586847773749,
   "beta_norm": 0.00010980771312097049,
   "P_obs": 7.55298193539245,
   "P_thm": 7.545718893322217,
   "I_obs": 10.613447280753375,
   "I_thm": 10.64582463899962,
   "S_obs": 0.5234174755207294,
   "S_thm": 0.5079117472280485
  }
 ]
}

In [3]:
if RUN:
    RESULTS = {'config': STORED['config'], 'rows': table()}
else:
    RESULTS = STORED

In [4]:
print(f"{'f':10s} {'r':>3s} {'indep.':>7s} | {'P obs':>6s} {'P thm':>6s} | {'I obs':>6s} {'I thm':>6s} | {'S obs':>6s} {'S thm':>6s} | {'|beta|':>7s} {'indep./S':>8s}")
for row in RESULTS['rows']:
    print(f"{row['f']:10s} {row['r']:3d} {row['independent']:7.2f} | {row['P_obs']:6.2f} {row['P_thm']:6.2f} | {row['I_obs']:6.2f} {row['I_thm']:6.2f} | "
          f"{row['S_obs']:6.3f} {row['S_thm']:6.3f} | {row['beta_norm']:7.4f} {row['independent'] / row['S_obs']:8.1f}")
print('\nStandard deviations in units of 1e-3. The last column is the ratio of the independent-sample value to the observed value in regime (S).')
print('Largest relative difference between observed and asymptotic values:')
for reg in 'PIS':
    d = {(row['f'], row['r']): abs(row[reg + '_obs'] / row[reg + '_thm'] - 1) for row in RESULTS['rows']}
    k = max(d, key=d.get)
    print(f"  regime ({reg}): {100 * d[k]:.0f}% for {k[0]} at r={k[1]}")

f            r  indep. |  P obs  P thm |  I obs  I thm |  S obs  S thm |  |beta| indep./S
sin x       20   14.70 |  15.14  14.72 |  20.80  20.89 |  0.750  0.678 |  0.2885     19.6
tanh x      20   14.04 |  14.43  14.04 |  19.74  19.81 |  0.741  0.765 |  0.0035     18.9
exp(-x^2)   20    7.55 |   7.64   7.57 |   9.98  10.26 |  1.935  1.951 |  0.0045      3.9
sin x       40   14.70 |  14.59  14.70 |  20.65  20.78 |  0.342  0.160 |  0.0841     43.0
tanh x      40   14.04 |  14.17  14.04 |  19.82  19.84 |  0.177  0.170 |  0.0016     79.5
exp(-x^2)   40    7.55 |   7.55   7.55 |  10.61  10.65 |  0.523  0.508 |  0.0001     14.4

Standard deviations in units of 1e-3. The last column is the ratio of the independent-sample value to the observed value in regime (S).
Largest relative difference between observed and asymptotic values:
  regime (P): 3% for sin x at r=20
  regime (I): 3% for exp(-x^2) at r=20
  regime (S): 114% for sin x at r=40
